In [76]:
import pyautogui as pyg
pyg.sleep(2)
basex=1475
basey=358
gap=40

for row in range(8):
    for col in range(8):
        x=basex+col*gap
        y=basey+col*gap

In [77]:
def gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的 (x, y) 座標（索引從 0 開始）。"""
    return (basex + c * gap, basey + r * gap)
    

In [78]:
def get_gem_color(r, c, im):
    """取得截圖 im 中第 r 列、第 c 欄寶石的 RGB 顏色。"""
    x, y = gem_position(r, c)
    return im.getpixel((x, y))

In [79]:
im=pyg.screenshot()
for i in range(8):
    x,y=gem_position(0,1)
    pyg.click(x,y,duration=1)
    print(get_gem_color(0,i,im))

(229, 27, 27)
(230, 27, 27)
(243, 15, 243)
(243, 15, 243)
(18, 133, 252)
(253, 249, 36)
(252, 27, 56)
(253, 249, 36)


In [80]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(1,0))

(1475, 358)
(1515, 358)
(1475, 398)


In [81]:
# Match-3：判斷第 (r, c) 顆寶石能否往上移動
def can_move_up(r, c, im):
    """若往上交換後形成三顆以上同色連線則回傳 True，否則回傳 False。"""
    if not (1 <= r < 8 and 0 <= c < 8):
        return False

    board = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]
    board[r][c], board[r - 1][c] = board[r - 1][c], board[r][c]

    def has_match_at(row, col):
        color = board[row][col]
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                rr, cc = row + dr * direction, col + dc * direction
                while (0 <= rr < 8 and 0 <= cc < 8
                       and board[rr][cc] == color):
                    count += 1
                    rr += dr * direction
                    cc += dc * direction
            if count >= 3:
                return True
        return False

    return has_match_at(r, c) or has_match_at(r - 1, c)

In [82]:
im=pyg.screenshot()
print(can_move_up(1,1,im))

True


In [ ]:
def _can_swap_make_match(r, c, im, dr, dc):
    """判斷寶石與指定方向的相鄰寶石交換後是否形成三連線。"""
    nr, nc = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    board = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]
    board[r][c], board[nr][nc] = board[nr][nc], board[r][c]

    def has_match_at(row, col):
        color = board[row][col]
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                rr = row + axis_r * direction
                cc = col + axis_c * direction
                while (0 <= rr < 8 and 0 <= cc < 8
                       and board[rr][cc] == color):
                    count += 1
                    rr += axis_r * direction
                    cc += axis_c * direction
            if count >= 3:
                return True
        return False

    return has_match_at(r, c) or has_match_at(nr, nc)


def can_move_down(r, c, im):
    """判斷 (r, c) 的寶石能否往下移動。"""
    return _can_swap_make_match(r, c, im, 1, 0)


def can_move_left(r, c, im):
    """判斷 (r, c) 的寶石能否往左移動。"""
    return _can_swap_make_match(r, c, im, 0, -1)


def can_move_right(r, c, im):
    """判斷 (r, c) 的寶石能否往右移動。"""
    return _can_swap_make_match(r, c, im, 0, 1)